# 2장 — 보폭의 과학: 학습률 📏 (핵심)

수축 인자 |1 − lr·c| 로 모든 운명이 결정됨을 코드로 확인하고, 워밍업의 미스터리(경계 1.00)를
풀고, 실전 요령(lr 스위프)까지 돌려 봅니다.

**이 노트북의 목표**
1. 수축 인자 공식이 실제 시뮬레이션과 정확히 일치함을 확인 (경계 2/c)
2. lr = 1/c 의 마법 — 한 방 도착
3. 곡률 비공개 골짜기에서 로그 스케일 lr 스위프로 좋은 보폭 찾기

In [ ]:
# 🤖 AI 튜터 — 막히면 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch2')

## 1. 실험 ① — 수축 인자 공식 검증

이차 골짜기 L = (c/2)(w−t)² 에서 한 스텝: **새 오차 = (1 − lr·c) × 지금 오차**.
공식이 예언하는 운명과 실제 시뮬레이션이 일치하는지 전수 대조합니다.

In [ ]:
import numpy as np

def simulate(lr, c, steps=40, err0=8.0):
    """이차 골짜기 경사하강 — 오차의 절대값 반환 (결정적)."""
    err = err0
    for _ in range(steps):
        err = err - lr * c * err          # w ← w − lr·∇L, 오차 기준으로 정리한 꼴
    return abs(err)

def fate_of(final, err0=8.0):
    """실제 시뮬 결과만 보고 판정 — 공식을 쓰지 않는다 (공정한 대조)."""
    if abs(final - err0) < 1e-6:
        return '진동'
    return '폭발' if final > err0 else '수렴'

print(' 곡률 c   lr     |1−lr·c|   공식의 예언   실제(40스텝)')
for c in [0.5, 2.0, 4.0]:
    for lr in [0.1, 1/c, 1.8/c, 2/c, 2.2/c]:
        f = abs(1 - lr * c)
        predict = '수렴' if f < 1 else ('진동' if abs(f - 1) < 1e-12 else '폭발')
        actual = fate_of(simulate(lr, c))
        mark = '✓' if predict == actual else '✗'
        print(f'  {c:5.1f}  {lr:5.2f}   {f:6.2f}     {predict}          {actual}  {mark}')
        assert predict == actual, '공식의 예언과 실제가 달라선 안 된다'

# 워밍업의 미스터리 — (w−3)² 은 c=2, 경계 = 2/2 = 1.00
# (0.99 는 수축 인자 0.98 — 수렴은 확실하지만 느려서 넉넉히 500스텝으로 확인)
assert simulate(0.99, 2.0, steps=500) < 1e-3, 'c=2: lr 0.99 는 (느리지만) 수렴'
assert abs(simulate(1.00, 2.0, steps=500) - 8) < 1e-9, 'c=2: lr 1.00 은 영원한 진동 — 워밍업의 그 경계'
assert simulate(1.01, 2.0, steps=40) > 8, 'c=2: lr 1.01 은 폭발'
print('\n✔ 확인: 경계 = 2/c — 워밍업 골짜기(c=2)의 미스터리 1.00 이 공식에서 나온다')

## 2. 실험 ② — lr = 1/c 의 마법: 한 방 도착

In [ ]:
for c in [0.5, 2.0, 4.0]:
    one = simulate(1/c, c, steps=1)
    print(f'곡률 {c}: lr = 1/c = {1/c:.2f} → 단 1스텝 후 오차 {one:.10f}')
    assert one < 1e-12, 'lr = 1/c 는 이차 골짜기에서 한 방 도착'

# 속도 비교 — 수축 인자 0.9 vs 0.5 의 20스텝
print(f'\n인자 0.9 → 20스텝 후 남은 비율 {0.9**20:.4f} (12%)')
print(f'인자 0.5 → 20스텝 후 남은 비율 {0.5**20:.2e} (백만분의 일)')
assert 0.5**20 < 0.9**20 / 1e4, '매 스텝의 작은 배율 차이가 거듭제곱으로 벌어진다'
print('✔ 확인: 속도의 정체도 수축 인자 — 0에 가까울수록 빠르다 (경계 2/c 는 넘지 말 것)')

## 3. 실험 ③ — 실전: 곡률 비공개 골짜기의 lr 스위프

실전에서는 c 를 모릅니다. 로그 스케일 후보를 10스텝씩 시험해 자릿수부터 좁힙니다.

In [ ]:
HIDDEN_C = 0.8   # (웹 실험과 같은 비공개 지형)

print(' lr      10스텝 후 손실     판정')
results = {}
for lr in [0.01, 0.03, 0.1, 0.3, 1.0, 3.0]:
    err = simulate(lr, HIDDEN_C, steps=10)
    loss = (HIDDEN_C / 2) * err**2
    results[lr] = loss
    start = (HIDDEN_C / 2) * 64
    fate = '💥 폭발' if loss > start else ('✅ 수렴' if loss < 0.01 else '🐢 느림')
    print(f'  {lr:5.2f}   {loss:15.6f}   {fate}')

best = min(results, key=results.get)
print(f'\n최고 성적: lr = {best}')
assert best == 1.0, '1.0 이 최고 — 한 방 도착 1/c = 1.25 에 가장 가까운 후보'
assert results[3.0] > (HIDDEN_C/2)*64, '3.0 은 폭발 — 경계 2/c = 2.5 를 넘었으므로'

# 경계 2.5 를 실험으로 확정
assert simulate(2.49, HIDDEN_C) < 8, '경계 바로 아래는 수렴'
assert simulate(2.51, HIDDEN_C) > 8, '경계 바로 위는 폭발'
print('✔ 확인: 경계는 2/0.8 = 2.5 — 스위프가 찾은 결과와 공식이 일치 (로그 스케일 = 자릿수부터)')

## 4. 관찰 정리 + 🤖 AI 연결

- 한 스텝 = 오차 × **수축 인자 |1 − lr·c|** — 0이면 한 방(lr=1/c), 1이 경계(lr=2/c), 넘으면 폭발.
- **가파를수록(c↑) 경계(2/c)는 내려온다** — 실전 LLM 의 lr 이 2e-5 처럼 작은 이유.
- 속도도 인자가 결정: 0.5²⁰ ≈ 10⁻⁶ vs 0.9²⁰ ≈ 0.12 — 수렴 속도 = 훈련 비용.
- 실전(곡률 비공개)의 요령: **로그 스케일 lr 스위프** — 자릿수 먼저, 미세 조정 나중.
- 남은 문제 두 가지: 방향마다 다른 곡률(→ 5장 Adam), 훈련 중 변하는 곡률(→ 6장 스케줄).

**다음 챕터**: 3장 — 흔들리는 나침반. 지금까지 기울기가 정확하다고 가정했지만, 실전의 기울기는
데이터 일부(미니배치)로 계산한 **표본 추정치**입니다 — 그리고 또 만나는 1/√n (3부 7장).